# BirdCLEF 2026 — Two-Stage SED + Species Classifier (Pipeline 05)

This pipeline splits the problem into two distinct stages to combat extreme background noise:
1. **Sound Event Detection (SED)**: A lightweight CRNN trained to detect *any* acoustic activity (Bird vs. Silence/Noise).
2. **Species Classifier**: A heavier EfficientNet-B0 trained specifically to differentiate species *only* within active regions.

By filtering out inactive regions first, we massively reduce false positives from wind, rain, and environmental noise.

## 1. Setup & Imports

In [ ]:
import os, gc, sys, math, time, glob, random, ast
import numpy as np, pandas as pd
from tqdm.auto import tqdm
import soundfile as sf
from pathlib import Path
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
import torchaudio.transforms as T
import timm
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')

## 2. Configuration

In [ ]:
class Config:
    ROOT_DIR       = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_CSV      = os.path.join(ROOT_DIR, 'train.csv')
    TRAIN_AUDIO_DIR = os.path.join(ROOT_DIR, 'train_audio')
    SOUNDSCAPE_CSV = os.path.join(ROOT_DIR, 'train_soundscapes_labels.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')

    MODEL_DIR = Path('/kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth')

    SR         = 32000
    WINDOW_SECONDS = 5

    N_MELS     = 128
    N_FFT      = 2048
    HOP_LENGTH = 512
    FMIN       = 20
    FMAX       = 16000

    # SED specific config
    SED_EPOCHS = 10
    SED_BATCH_SIZE = 64
    SED_LR = 1e-3

    # Species Classifier config
    SPEC_EPOCHS = 15
    SPEC_BATCH_SIZE = 32
    SPEC_LR = 1e-4

    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 0
    MODEL_NAME = 'tf_efficientnet_b0'
    SEED = 42
    NUM_CLASSES = 0

CFG = Config()

def seed_everything(seed):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
seed_everything(CFG.SEED)

train_df   = pd.read_csv(CFG.TRAIN_CSV)
ss_df      = pd.read_csv(CFG.SOUNDSCAPE_CSV)
sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
label_to_id = {label: i for i, label in enumerate(submission_labels)}
train_df['label_id'] = train_df['primary_label'].map(label_to_id)
CFG.NUM_CLASSES = len(submission_labels)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 3. SED Dataset (Binary: Bird vs No-Bird)

In [ ]:
class SEDDataset(Dataset):
    """Dataset for training the binary Sound Event Detector. Uses soundscapes heavily."""
    def __init__(self, df_ss, audio_dir, is_train=True):
        self.df = df_ss.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.is_train = is_train
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        self.mel_transform = T.MelSpectrogram(sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH, n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX)
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row['filename'])
        h, m, s = map(int, row['start'].split(':'))
        start_s = (h * 3600 + m * 60 + s) * CFG.SR
        try:
            y, _ = sf.read(path, start=start_s, stop=start_s + self.window_samples, always_2d=True)
            y = y.mean(axis=1)
            if len(y) < self.window_samples: y = np.pad(y, (0, self.window_samples - len(y)))
        except:
            y = np.zeros(self.window_samples)
        
        if self.is_train and random.random() < 0.5: y = y + 0.005 * np.random.randn(len(y))
        
        y_t = torch.tensor(y, dtype=torch.float32)
        mel = self.amplitude_to_db(self.mel_transform(y_t))
        mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
        img = torch.stack([mel, mel, mel])
        
        # Target: 1 if any bird is present, 0 if 'nocall' or no valid label
        has_bird = 0.0
        labels = str(row['primary_label']).split(';')
        for lbl in labels:
            if lbl in label_to_id:
                has_bird = 1.0
                break
        
        target = torch.tensor([has_bird], dtype=torch.float32)
        return img, target


## 4. Species Dataset (Multilabel)

In [ ]:
class SpeciesDataset(Dataset):
    """Standard multilabel dataset for Phase 2, focusing on active regions."""
    def __init__(self, df_clips, df_ss, clip_dir, ss_dir, is_train=True):
        # For soundscapes, we ONLY want active regions for training the species classifier
        active_ss = df_ss[df_ss['primary_label'].apply(lambda x: any(l in label_to_id for l in str(x).split(';')))]
        
        # Standardize columns so we can concat
        clips = df_clips[['filename', 'primary_label', 'secondary_labels', 'label_id']].copy()
        clips['is_soundscape'] = False
        
        ss = active_ss[['filename', 'primary_label', 'start']].copy()
        ss['secondary_labels'] = ''
        ss['label_id'] = -1 # Handled dynamically
        ss['is_soundscape'] = True
        
        self.df = pd.concat([clips, ss], ignore_index=True)
        self.clip_dir = clip_dir
        self.ss_dir = ss_dir
        self.is_train = is_train
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        self.mel_transform = T.MelSpectrogram(sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH, n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX)
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if row['is_soundscape']:
            path = os.path.join(self.ss_dir, row['filename'])
            h, m, s = map(int, row['start'].split(':'))
            start_s = (h * 3600 + m * 60 + s) * CFG.SR
            try:
                y, _ = sf.read(path, start=start_s, stop=start_s + self.window_samples, always_2d=True)
                y = y.mean(axis=1)
            except: y = np.zeros(self.window_samples)
        else:
            path = os.path.join(self.clip_dir, row['filename'])
            try:
                total = sf.info(path).frames
                if total > self.window_samples:
                    start = random.randint(0, total - self.window_samples) if self.is_train else 0
                    y, _ = sf.read(path, start=start, frames=self.window_samples, always_2d=True)
                else:
                    y, _ = sf.read(path, always_2d=True)
                y = y.mean(axis=1)
            except: y = np.zeros(self.window_samples)
            
        if len(y) < self.window_samples: y = np.pad(y, (0, self.window_samples - len(y)))
        if self.is_train and random.random() < 0.5: y = y + 0.005 * np.random.randn(len(y))
        
        y_t = torch.tensor(y, dtype=torch.float32)
        mel = self.amplitude_to_db(self.mel_transform(y_t))
        mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
        img = torch.stack([mel, mel, mel])
        
        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        for lbl in str(row['primary_label']).split(';'):
            if lbl in label_to_id: target[label_to_id[lbl]] = 1.0
            
        if not row['is_soundscape'] and 'secondary_labels' in row and pd.notna(row['secondary_labels']):
            try:
                for sl in ast.literal_eval(row['secondary_labels']):
                    if sl in label_to_id: target[label_to_id[sl]] = 1.0
            except: pass
            
        return img, target


## 5. Models (SED CRNN & Species EfficientNet)

In [ ]:
class SED_CRNN(nn.Module):
    """Lightweight CRNN for Binary Sound Event Detection"""
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.rnn = nn.GRU(input_size=128 * (CFG.N_MELS // 16), hidden_size=128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, 1)
        
    def forward(self, x):
        x = self.cnn(x) # (B, 128, F', T')
        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2).contiguous() # (B, T', C, F')
        x = x.view(B, T, C * F)
        x, _ = self.rnn(x)
        # Global max pooling over time for clip-level SED prediction
        x, _ = torch.max(x, dim=1)
        return self.fc(x)

class SpeciesClassifier(nn.Module):
    """EfficientNet for species classification on active segments"""
    def __init__(self, model_name, num_classes, model_path=None):
        super().__init__()
        if model_path is not None and Path(model_path).exists():
            self.backbone = timm.create_model(model_name, checkpoint_path=model_path, pretrained=False, in_chans=3)
        else:
            self.backbone = timm.create_model(model_name, pretrained=False, in_chans=3)
        
        if 'efficientnet' in model_name:
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        else:
            in_features = self.backbone.get_classifier().in_features
            self.backbone.reset_classifier(0)
            
        self.head = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.head(self.backbone(x))


## 6. Training Logic

In [ ]:
def train_epoch_sed(model, loader, optimizer, criterion, scaler, device):
    model.train()
    epoch_loss = 0.0
    for images, targets in tqdm(loader, desc='SED Train', leave=False):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def valid_epoch_sed(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0.0
    preds, true_targets = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='SED Valid', leave=False):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            epoch_loss += loss.item()
            preds.append(torch.sigmoid(outputs).cpu().numpy())
            true_targets.append(targets.cpu().numpy())
    preds = np.concatenate(preds)
    true_targets = np.concatenate(true_targets)
    # Binary AUC for SED
    try: auc = roc_auc_score(true_targets, preds)
    except: auc = 0.0
    return epoch_loss / len(loader), auc

def train_epoch_spec(model, loader, optimizer, criterion, scaler, device):
    model.train()
    epoch_loss = 0.0
    for images, targets in tqdm(loader, desc='Spec Train', leave=False):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def valid_epoch_spec(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0.0
    preds, true_targets = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Spec Valid', leave=False):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            epoch_loss += loss.item()
            preds.append(torch.sigmoid(outputs).cpu().numpy())
            true_targets.append(targets.cpu().numpy())
    preds = np.concatenate(preds)
    true_targets = np.concatenate(true_targets)
    auc_scores = []
    for i in range(CFG.NUM_CLASSES):
        if len(np.unique(true_targets[:, i])) > 1:
            auc_scores.append(roc_auc_score(true_targets[:, i], preds[:, i]))
    final_auc = np.mean(auc_scores) if auc_scores else 0.0
    return epoch_loss / len(loader), final_auc


## 7. Main Execution

In [ ]:
train_ss_df, valid_ss_df = train_test_split(ss_df, test_size=0.2, random_state=CFG.SEED)

# ── Phase 1: Train SED Model ───────────────────────────────
print('\n>>> PHASE 1: Training Binary Sound Event Detector (SED) <<<')
sed_train_ds = SEDDataset(train_ss_df, CFG.SOUNDSCAPE_DIR, is_train=True)
sed_valid_ds = SEDDataset(valid_ss_df, CFG.SOUNDSCAPE_DIR, is_train=False)
sed_train_loader = DataLoader(sed_train_ds, batch_size=CFG.SED_BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS)
sed_valid_loader = DataLoader(sed_valid_ds, batch_size=CFG.SED_BATCH_SIZE, shuffle=False)

sed_model = SED_CRNN().to(device)
sed_optimizer = optim.AdamW(sed_model.parameters(), lr=CFG.SED_LR)
sed_criterion = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler()

best_sed_auc = 0
for epoch in range(1, CFG.SED_EPOCHS + 1):
    t_loss = train_epoch_sed(sed_model, sed_train_loader, sed_optimizer, sed_criterion, scaler, device)
    v_loss, v_auc = valid_epoch_sed(sed_model, sed_valid_loader, sed_criterion, device)
    print(f'SED Epoch {epoch} | Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val AUC: {v_auc:.4f}')
    if v_auc > best_sed_auc:
        best_sed_auc = v_auc
        torch.save(sed_model.state_dict(), 'best_sed_crnn.pth')
        print('  -> Saved best SED model.')

# ── Phase 2: Train Species Classifier ─────────────────────────
print('\n>>> PHASE 2: Training Species Classifier (on active regions only) <<<')
df = train_df.copy()
counts = df['label_id'].value_counts()
rare_birds = counts[counts < 2].index.tolist()
if rare_birds:
    df = pd.concat([df, df[df['label_id'].isin(rare_birds)]], ignore_index=True)
train_df_clips, valid_df_clips = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=CFG.SEED)

spec_train_ds = SpeciesDataset(train_df_clips, train_ss_df, CFG.TRAIN_AUDIO_DIR, CFG.SOUNDSCAPE_DIR, is_train=True)
spec_valid_ds = SpeciesDataset(valid_df_clips, valid_ss_df, CFG.TRAIN_AUDIO_DIR, CFG.SOUNDSCAPE_DIR, is_train=False)
spec_train_loader = DataLoader(spec_train_ds, batch_size=CFG.SPEC_BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS)
spec_valid_loader = DataLoader(spec_valid_ds, batch_size=CFG.SPEC_BATCH_SIZE, shuffle=False)

spec_model = SpeciesClassifier(CFG.MODEL_NAME, CFG.NUM_CLASSES, CFG.MODEL_DIR).to(device)
spec_optimizer = optim.AdamW(spec_model.parameters(), lr=CFG.SPEC_LR, weight_decay=CFG.WEIGHT_DECAY)
spec_criterion = nn.BCEWithLogitsLoss()

best_spec_auc = 0
for epoch in range(1, CFG.SPEC_EPOCHS + 1):
    t_loss = train_epoch_spec(spec_model, spec_train_loader, spec_optimizer, spec_criterion, scaler, device)
    _, v_auc = valid_epoch_spec(spec_model, spec_valid_loader, spec_criterion, device)
    print(f'Spec Epoch {epoch} | Loss: {t_loss:.4f} | Val AUC: {v_auc:.4f}')
    if v_auc > best_spec_auc:
        best_spec_auc = v_auc
        torch.save(spec_model.state_dict(), 'best_species_classifier.pth')
        print('  -> Saved best Species Classifier model.')
